# Programming and Tools for AI (/Online)
# James McDermott, University of Galway, 2024
# CT5132/CT5148
# Assignment 2: DNA 

# Solutions


# Classifying DNA Sequences using $n$-grams

In this assignment, we'll put together a classification method specialised to mining DNA sequences, in the style of Scikit-Learn. We'll use components from Scikit-Learn itself and Numpy. You need to know a little machine learning, specifically the concept of *k-nearest neighbours classification*, and the Scikit-Learn object-oriented API. You don't need to know any genetics (and we won't be using real data).

Let's suppose we have a dataset of DNA sequences, drawn from different species, like this:

* `ATCGAGGTCCACGCTCGAACAACGATATGC`: mushroom
* `TATGCGTGTTACCACTTT` rose
* `ACTGATTGCACGGATGTATAGCGAACT`: hogweed

DNA on our world is always composed of a sequence of four amino acids which are named `A`, `C`, `G`, and `T`.

As you can see, they are very short, so they are not the *complete* DNA of these species. And they are of different lengths, so our typical machine learning methods based on feature vectors don't easily apply.

Also, commonalities between sequences might be mis-aligned, eg these two sequences are pretty similar even though comparing letter-by-letter they seem dissimilar:

* `AAACGTAC`
* `ACGTAC`


<img src=Mammoth-brittanica.png width=50%>

(Image from https://www.britannica.com/animal/mammoth-extinct-mammal .)


Now, suppose we are interested in the diets of ancient plant-eating mammoths. We manage to extract a DNA sample from the stomach of a mammoth which was frozen in ice:

* `ACGATCGATCGATACAA`: unknown.


We want to *predict*: what type of plant might it be from? 

It doesn't match any of our known samples exactly. But here's an approach, based on *n-grams*: 

* We extract *n-grams*, ie for a value of $n$ such as $n=4$, we get the following *n-grams*: 
* `ACGA`, `CGAT`, `GATC`, `ATCG`, etc.
* We count how often each occurs, eg:
* `GATC`: 2, `ACAA`: 1, etc. This is a *multi-set* (each item can occur many times: we count how often each occurs).
* Then we do the same for the known sequences, and calculate how much overlap our unknown species' *n-grams* have with the known ones, eg:
* `ACAA` is in common with *mushroom*, but `GATC` is not present in *mushroom*.
* Specifically, we'll use a metric called *intersection over union*, which we'll define later.
* That tells us which existing sequences our sequence is most similar to.
* We then use *k-nearest neighbours* to make a prediction.
* For example, in the image, with $k=1$, we would predict the unknown sequence is *rose*. 
* Or with $k=3$, we would predict the unknown sequence is *hogweed*.

![](n-gram-DNA-plants.drawio.png)


This notebook will walk us through all the steps, with doctests and examples for each. Most of the code needed is based on something we've seen in class.


In [1]:
import sys
assert sys.version_info >= (3, 7), "This notebook requires Python 3.7+"

from collections import Counter
import doctest

import numpy as np

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

### Data

We have a file containing many sequences together with their species labels. We are given code to load it and do a train-test split:

In [2]:
def read_data(filename):
    X = []
    y = []
    for line in open(filename).readlines():
        xi, yi = line.strip().split(" ")
        X.append(xi)
        y.append(yi)
    return train_test_split(X, y, test_size=0.5, random_state=42) 

In [3]:
X_train, X_test, y_train, y_test = read_data("dna_sequences_data.txt")

**Question 1**. Write a function `count_ngrams` which accepts a sequence (eg a list, string, or tuple), extracts all *n-grams*, ie all subsequences of length `n`, and returns their number of occurrences in a `Counter`. Notice that if the input is a string, then each *n-gram* is represented as a string.

In [4]:
def count_ngrams(x, n):
    '''
The input can be a DNA string:
>>> count_ngrams("gactacgacacag", 3)
Counter({'gac': 2, 'aca': 2, 'act': 1, 'cta': 1, 'tac': 1, 
         'acg': 1, 'cga': 1, 'cac': 1, 'cag': 1})

If shorter than n, there will be no n-grams:
>>> count_ngrams("gactacgacacag", 17)
Counter()

It can even be empty:         
>>> count_ngrams("", 3)
Counter()

It can even be an ordinary (non-DNA) string:
>>> count_ngrams("abcdabcdefacd", 3)
Counter({'abc': 2, 'bcd': 2, 'cda': 1, 'dab': 1, 'cde': 1, 
         'def': 1, 'efa': 1, 'fac': 1, 'acd': 1})
    '''
    return Counter(x[i:i+n] for i in range(len(x) - n + 1))

In [5]:
doctest.run_docstring_examples(count_ngrams, globals(), optionflags=doctest.NORMALIZE_WHITESPACE)

**Question 2**. Suppose we have a list of sequences, `X`. Write a function which gets the ngrams for each sequence in `X`. Of course it should just use our `count_ngrams`. Notice that `X` could be *ragged*, ie not every sequence in `X` has to be the same length.

In [6]:
def count_ngrams_multi(X, n):
    '''
>>> count_ngrams_multi(["catgcatg",
...                     "acacaca",
...                     "gtaaagtaaa"], 3)
[Counter({'cat': 2, 'atg': 2, 'tgc': 1, 'gca': 1}), 
 Counter({'aca': 3, 'cac': 2}), 
 Counter({'gta': 2, 'taa': 2, 'aaa': 2, 'aag': 1, 'agt': 1})]
>>> count_ngrams_multi([[]], 3)
[Counter()]
    '''
    return [count_ngrams(x, n) for x in X]

In [7]:
# This runs our count_ngrams_multi doctests.
doctest.run_docstring_examples(count_ngrams_multi, globals(), optionflags=doctest.NORMALIZE_WHITESPACE)

**Question 3**. We're going to need a helper function `counter_total` which measures the "total size" of a `Counter` by counting the total of its individual counts.

In [8]:
def counter_total(c):
    '''
    Sum of the counts in a counter c
    
    >>> counter_total(Counter())
    0
    >>> counter_total(Counter({'a': 1, 'b': 3}))
    4
    '''
    return c.total() # in Python 3.7, 3.8, 3.9, you can use sum(Counter.values()) instead

In [9]:
doctest.run_docstring_examples(counter_total, globals(), optionflags=doctest.NORMALIZE_WHITESPACE)

**Question 4**. Next, we want to measure the *dissimilarity* between two sequences, `X` and `Y`.  

Since we already have a method for representing a sequence as a `Counter` of n-grams, we only need a method for measuring the dissimilarity of two `Counter`s, $A$ and $B$. One main approach is the *Jaccard dissimilarity*. It's defined as:

$$\mathrm{JD}(A, B) = 1 - \frac{|A \cap B|}{|A \cup B|}$$

(Source: https://en.wikipedia.org/wiki/Jaccard_index.)

In other words, $\mathrm{JD}$ is low (dissimilarity low => similarity high) if there are many $n$-grams in common between `X` and `Y`. 

The value we want to calculate is the **dissimilarity** (the opposite of the **similarity**) because our *k-nearest neighbours* function later on needs a dissimilarity.

By the way, if you Google for "python jaccard distance" you'll find a lot of results which are not what we want, so be careful.

Hint: the operators `&` and `|` are defined for `Counter`s and will be useful.

In [10]:
def JD(A, B):
    '''Jaccard dissimilarity on multisets, represented by Counters.
    A multiset is like a set, as it's composed of unordered objects, 
    but every item can occur more than once. It counts *how many times*
    each item occurs.
    
    Two are identical:
    >>> JD(Counter({'cat': 2, 'atg': 2, 'tgc': 1, 'gca': 1}), 
    ...    Counter({'cat': 2, 'atg': 2, 'tgc': 1, 'gca': 1}))
    0.0
    
    Two are similar but not identical:
    >>> JD(Counter({'cat': 1, 'atg': 1, 'tgc': 1, 'gca': 1}), 
    ...    Counter({'cat': 1, 'atg': 1, 'tgc': 1, 'gcg': 1}))
    0.4
    
    Worked example: in the above two Counters, their intersection is size 3 and their union is size 5.
    Therefore, the JD is 1 - 3/5 = 0.4.

    Notice that defining JD over sets would not be correct. The
    following Counters are the same if viewed as sets (they both have
    the same items) so JD would equal 0. But the correct implementation
    views them as multisets, ie it takes account of the number of each item. 
    >>> JD(Counter({'cat': 2, 'atg': 1}), 
    ...    Counter({'cat': 1, 'atg': 2}))
    0.5

    Two are totally different:
    >>> JD(Counter({'cat': 2, 'atg': 2, 'tgc': 1, 'gca': 1}), 
    ...    Counter({'tca': 2, 'atc': 2, 'tga': 1, 'gct': 1}))
    1.0
    
    Note, the JD code doesn't have to work only on n-gram Counters. The same code should work on any Counters:
    >>> JD(Counter([0, 0, 0, 1, 1, 2, 3]), 
    ...    Counter([0, 0, 0, 1, 1, 2, 4]))
    0.25

    '''
    intersection = A & B
    union = A | B
    return 1 - counter_total(intersection) / counter_total(union)

In [11]:
doctest.run_docstring_examples(JD, globals(), optionflags=doctest.NORMALIZE_WHITESPACE)

**Question 5**. In $k$-nearest neighbours, we need to be able to calculate the dissimilarity between all items in a list (the list of query points) and all items in another list (the list of training points).

Eg, if our query list has 2 points, and the training data has 3 points, we need to calculate a distance matrix of shape (2, 3). Let's assume the user to our function will pass in a metric which acts like our `JD`, that is it accepts two objects and returns a float.

In [12]:
def distance_matrix(XA, XB, metric):
    """
    >>> XA = [Counter([1,1,1,2]), Counter([1,1,2,3])]
    >>> XB = [Counter([1,1,1,2]), Counter([1,1,2,4]), Counter([1,2,4])]
    >>> distance_matrix(XA, XB, JD)
    array([[0. , 0.4, 0.6],
           [0.4, 0.4, 0.6]])

    Location (0, 0) gives value 0.0 because XA[0] is equal to XB[0].
    Location (1, 2) gives value 0.6 because XA[1] and XB[2] have intersection of size 2 and union of size 5. JD gives 1 - 2/5 = 0.6.
    """
    result = np.zeros((len(XA), len(XB)))
    for i, xa in enumerate(XA):
        for j, xb in enumerate(XB):
            result[i, j] = metric(xa, xb)
    return result

In [13]:
doctest.run_docstring_examples(distance_matrix, globals(), optionflags=doctest.NORMALIZE_WHITESPACE)

The next component needed is: given an array, we want to be able to find out in each row which are the indices which hold the smallest values. This function is given below, you don't need to write anything.

In [14]:
def find_k_min_indices(X, k):
    """
    >>> X = np.array([[0.57, 0.53, 0.03, 0.19, 0.71],
    ...               [0.51, 0.91, 0.41, 0.27, 0.02],
    ...               [0.30, 0.37, 0.38, 0.57, 0.39],
    ...               [0.05, 0.37, 0.38, 0.83, 0.13]])
    >>> find_k_min_indices(X, 3)
    array([[2, 3, 1],
           [4, 3, 2],
           [0, 1, 2],
           [0, 4, 1]])
    >>> find_k_min_indices(X, 1) # k=1 makes sense
    array([[2],
           [4],
           [0],
           [0]])
    >>> find_k_min_indices(X, 10) # k=10 makes no sense as there are only 5 items per row
    Traceback (most recent call last):
    ...
    ValueError
    """

    # THIS IS COMPLETE: YOU DON'T NEED TO WRITE ANYTHING
    
    if k > X.shape[1]: raise ValueError
    
    # Get the indices that would sort each row of X
    sorted_indices = np.argsort(X, axis=1)
    
    # Return the first k indices for each row
    return sorted_indices[:, :k]

In [15]:
doctest.run_docstring_examples(find_k_min_indices, globals(), optionflags=doctest.NORMALIZE_WHITESPACE)

**Question 6**. Now, we're ready to put everything together. Create a class `NGramsKNN` which inherits from `BaseEstimator` and `ClassifierMixin`:

* It should have an `__init__` function which stores the values of $n$ and $k$. 
* It should have a `fit` method, where the input `X` is a list or array of sequences and the input `y` is a list or array of labels. (In our case they will be string labels, such as `['mushroom', 'rose', 'rose', ...]`, but in general labels could be integers or something else.) The `fit` method should store the set of labels observed during training as `self.classes_`.
* It should have a `predict` method, which predicts by majority vote of the $k$ closest items to the query point. `predict` should calculate a distance matrix and then use `find_k_min_indices`. It should also use our `vote` function, given below. The input to `predict` is a new list or array of sequences `X` and it should return an array of predicted labels.

In [16]:
def vote(x):
    # easy way to vote. ties are broken arbitrarily.
    # eg Counter('aaaabbbccd').most_common(1) -> (('a', 4),)
    # so [0][0] -> 'a'
    # THIS IS COMPLETE. NO NEED TO WRITE ANYTHING.
    return Counter(x).most_common(1)[0][0] 

class NGramsKNN(BaseEstimator, ClassifierMixin):
    """
    A class representing k-nearest neighbours classification, using Jaccard dissimilarity
    as a measure of distance ("nearest") on counters of n-grams
    extracted from the sequences X.

    Doctests in the next cell.
    """
    def __init__(self, n=3, k=3):
        super().__init__()
        self.n = n
        self.k = k

    def fit(self, X, y):
        # find and store the n-grams as one Counter per point in training set X
        self.X_counters = count_ngrams_multi(X, self.n)
        self.y = np.array(y)
        self.classes_ = set(y)
        return self
    
    def predict(self, X):
        # find and store the n-grams as one Counter per point in query set X
        X_counters = count_ngrams_multi(X, self.n)

        # distance matrix: one row per query point, one column per training set point
        D = distance_matrix(X_counters, self.X_counters, JD)

        # for each query point, find the k nearest training points
        k_nearest_indices = find_k_min_indices(D, self.k)

        result = []
        for idxs in k_nearest_indices: # for each query point
            ys = self.y[idxs] # the y labels of the k nearest training points
            result.append(vote(ys)) # majority vote
        return np.array(result)


In [22]:
def test_NGramsKNN():
    """
    There should be a parameter n with default value as follows:
    >>> NGramsKNN().n
    3

    There should be a parameter k with default value as follows:
    >>> NGramsKNN().k
    3

    We can create the object, and if we don't over-ride default 
    parameters, no parameter values are shown. Hint: in __init__, use super().
    >>> NGramsKNN()
    NGramsKNN()
    
    We can over-ride the default values, and then they are printed, eg:
    >>> NGramsKNN(k=5, n=4)
    NGramsKNN(k=5, n=4)

    Don't forget, fit() should return self:
    >>> X = ['abcdef', 'abcde', 'fedcbaaba', 'abcdefg', 'bbcde', 'ffdccba']
    >>> y = [1, 0, 1, 0, 0, 1]
    >>> NGramsKNN(n=2, k=1).fit(X, y)
    NGramsKNN(k=1, n=2)
    >>> NGramsKNN(n=2, k=1).fit(X, y).predict(['abcdef', 'fedcba'])
    array([1, 1])
    >>> NGramsKNN(n=2, k=3).fit(X, y).predict(['abcdef', 'fedcba'])
    array([0, 1])


    Before fitting, the object should not have the trailing-underscore attribute classes_:
    >>> nk = NGramsKNN(n=2, k=1)
    >>> hasattr(nk, "classes_")
    False

    After fitting, the object should have the trailing-underscore attribute classes_:
    >>> nk = NGramsKNN(n=2, k=1).fit(X, y)
    >>> hasattr(nk, "classes_")
    True
    """
    pass

In [23]:
# currently doctests in classes are broken (https://github.com/python/cpython/pull/113335)
# so we work around it by putting our tests in a function.
doctest.run_docstring_examples(test_NGramsKNN, globals(), optionflags=doctest.NORMALIZE_WHITESPACE)

**Question 7**: let's check out how the score changes for different $k$ and different $n$. We should end up with a table something like this (your figures and formatting could be different):

```
n  k  score
3  1  0.48
3  3  0.5
3  5  0.48
4  1  0.62
4  3  0.62
4  5  0.7
5  1  0.68
5  3  0.74
5  5  0.7
6  1  0.74
6  3  0.72
6  5  0.74
8  1  0.64
8  3  0.72
8  5  0.72
```

In [25]:
# my solution
print("n  k  score")
for n in [3, 4, 5, 6, 8]:
    for k in [1, 3, 5]:
        clf = NGramsKNN(n, k)
        clf.fit(X_train, y_train)
        print(f"{n}  {k}  {clf.score(X_test, y_test)}")

n  k  score
3  1  0.48
3  3  0.5
3  5  0.48
4  1  0.62
4  3  0.62
4  5  0.7
5  1  0.68
5  3  0.74
5  5  0.7
6  1  0.74
6  3  0.72
6  5  0.74
8  1  0.64
8  3  0.72
8  5  0.72


**Question 8**. Finally, here's the unknown query sequence which was extracted from the mammoth. Write code to find out what species it is likely to be.


In [20]:
unknown = "CACGACTGCCAATAGCCAAGCCA"

In [21]:
clf = NGramsKNN(5, 3)
clf.fit(X_train, y_train)
print(clf.predict([unknown]))

['grass']
